In [29]:
import pandas as pd
import plotly.express as px
import glob

# Load the data from multiple CSV files
files = glob.glob('./data/CARGA_ENERGIA_*.csv')
data_list = [pd.read_csv(file) for file in files]
data = pd.concat(data_list, ignore_index=True)

# Split the single column into multiple columns
data[['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cargaenergiamwmed']] = data['id_subsistema;nom_subsistema;din_instante;val_cargaenergiamwmed'].str.split(';', expand=True)

# Convert the 'din_instante' column to datetime
data['din_instante'] = pd.to_datetime(data['din_instante'])

# Convert 'val_cargaenergiamwmed' to numeric
data['val_cargaenergiamwmed'] = pd.to_numeric(data['val_cargaenergiamwmed'])

# Define the coordinates for each region
coordinates = {
    'Norte': {'lat': -3.1190275, 'lon': -60.0217314},
    'Nordeste': {'lat': -8.0475622, 'lon': -34.877},
    'Sul': {'lat': -30.0346471, 'lon': -51.2176584},
    'Sudeste/Centro-Oeste': {'lat': -23.5506507, 'lon': -46.6333824}
}

# Add latitude and longitude to the dataframe
data['lat'] = data['nom_subsistema'].map(lambda x: coordinates[x]['lat'])
data['lon'] = data['nom_subsistema'].map(lambda x: coordinates[x]['lon'])

# Aggregate the data by region and sum the values
agg_data = data.groupby(['nom_subsistema', 'lat', 'lon'], as_index=False)['val_cargaenergiamwmed'].sum()

# Create the heatmap with larger bubbles and display the value
fig = px.scatter_geo(
    agg_data,
    lat='lat',
    lon='lon',
    size='val_cargaenergiamwmed',
    color='val_cargaenergiamwmed',
    hover_name='nom_subsistema',
    hover_data={'val_cargaenergiamwmed': True},
    title='Sum of Carga Média MW by Region (2015-2025)',
    scope='south america',
    size_max=35  # Increase the maximum size of the bubbles
)

fig.show()
# Calculate the mean for each region
mean_values = agg_data.groupby('nom_subsistema')['val_cargaenergiamwmed'].mean()

# Create the subtitle text
subtitle_text = 'Mean Carga Média MW by Region:\n' + '\n'.join([f"{region}: {mean:.2f}" for region, mean in mean_values.items()])

# Update the figure with the subtitle
fig.update_layout(
    annotations=[
        dict(
            x=0.5,
            y=-0.1,
            xref='paper',
            yref='paper',
            text=subtitle_text,
            showarrow=False,
            font=dict(size=12)
        )
    ]
)

In [22]:
fig_bar = px.bar(
    agg_data,
    x='nom_subsistema',
    y='val_cargaenergiamwmed',
    title='Average Carga Média MW by Region',
    labels={'val_cargaenergiamwmed': 'Average Carga Média MW', 'nom_subsistema': 'Region'}
)

fig_bar.show()

In [10]:
import pandas as pd
import plotly.express as px
# Load all the data files
"""file_paths = [
    './data/CARGA_ENERGIA_2015.csv', './data/CARGA_ENERGIA_2016.csv', './data/CARGA_ENERGIA_2017.csv',
    './data/CARGA_ENERGIA_2018.csv', './data/CARGA_ENERGIA_2019.csv', './data/CARGA_ENERGIA_2020.csv',
    './data/CARGA_ENERGIA_2021.csv', './data/CARGA_ENERGIA_2022.csv', './data/CARGA_ENERGIA_2023.csv',
    './data/CARGA_ENERGIA_2024.csv', './data/CARGA_ENERGIA_2025.csv'
]"""

# Combine all the data into a single dataframe
file_paths = [f'./data/CARGA_ENERGIA_{year}.csv' for year in range(2015, 2026)]
yearly_data = pd.concat((pd.read_csv(file) for file in file_paths), ignore_index=True)

# Split the single column into multiple columns
yearly_data[['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cargaenergiamwmed']] = yearly_data['id_subsistema;nom_subsistema;din_instante;val_cargaenergiamwmed'].str.split(';', expand=True)

# Convert the 'din_instante' column to datetime
yearly_data['din_instante'] = pd.to_datetime(yearly_data['din_instante'])

# Convert 'val_cargaenergiamwmed' to numeric
yearly_data['val_cargaenergiamwmed'] = pd.to_numeric(yearly_data['val_cargaenergiamwmed'])

# Aggregate the data by year
yearly_data['year'] = yearly_data['din_instante'].dt.year
agg_yearly_data = yearly_data.groupby('year', as_index=False)['val_cargaenergiamwmed'].mean()

# Create the line plot
fig_yearly = px.line(
    agg_yearly_data,
    x='year',
    y='val_cargaenergiamwmed',
    title='Average Carga Média MW by Year',
    labels={'val_cargaenergiamwmed': 'Average Carga Média MW', 'year': 'Year'}
)

fig_yearly.show()
